# 02 · BEM tuning against monitored indoor temperatures

Tunes the baseline BEM of a building (**Sections 3.3.1 and 4.2**) and evaluates the agreement
between the simulated and the measured indoor temperature of the monitored classroom (A1.3 in
Building 10, B1.5 in Building 09). As in the error metrics of Table 3, the simulated
temperature is the Zone Mean Radiant Temperature of the classroom zone (wall-mounted sensors).

The tuning edits are listed in `data/<building>/<building>_calibration_settings.json`:
simulation timestep, day/night temperature setpoints as Schedule:Compact objects, always-on
availability of the HVAC systems, lighting and equipment schedules and power densities,
outdoor-air method and external blind geometry.

**Inputs**: baseline BEM (`models/<building>_bem_baseline.idf`, or the model regenerated by
`01_bim_to_bem` with `--baseline regenerated`), monitoring data (`data/monitoring/`), weather file.

**Outputs** (`results/calibration/`): tuned BEM, simulation results, error metrics of the
baseline (pre-tuning) and tuned (post-tuning) models, monthly comparison.

## 0. Set-up

In [ ]:
import json
import shutil
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
ROOT = next(p for p in (HERE, *HERE.parents) if (p / 'config.py').exists())
sys.path.insert(0, str(ROOT))
import config

import numpy as np
import pandas as pd
import plotly.express as px
from eppy.modeleditor import IDF

from digitmanpy.Operation import Eppy

In [ ]:
# Building to tune: 'building09' or 'building10'
BUILDING = config.cli_option('building', 'building10')
# Baseline BEM: 'archived' (models/, used in the paper) or 'regenerated' (results/bem/, from 01_bim_to_bem)
BASELINE = config.cli_option('baseline', 'archived')
# Run the EnergyPlus simulations (otherwise the results of a previous run are read)
RUN_SIMULATIONS = config.cli_option('simulate', 'yes') == 'yes'
# Simulated zone temperature compared with the sensor readings
SIMULATED_TEMPERATURE = 'Zone Mean Radiant Temperature'

settings = json.loads(config.building_file(BUILDING, 'calibration_settings.json').read_text())
monitored_room = config.MONITORED_ROOMS[BUILDING]

baseline_path = (config.MODELS_DIR if BASELINE == 'archived' else config.BEM_RESULTS_DIR) / f'{BUILDING}_bem_baseline.idf'
archived_calibrated_path = config.MODELS_DIR / f'{BUILDING}_bem_calibrated.idf'

output_dir = config.CALIBRATION_RESULTS_DIR
simulation_dir = output_dir / 'simulation'
figures_dir = output_dir / 'figures'
simulation_dir.mkdir(parents=True, exist_ok=True)
calibrated_path = output_dir / f'{BUILDING}_bem_calibrated.idf'

print(f"{config.BUILDINGS[BUILDING]['name']}: monitored room {monitored_room['room']} ({monitored_room['zone']})")

## 1. Tuning of the baseline BEM

In [ ]:
IDF.setiddname(str(config.IDD_V24_1))
idf = IDF(str(baseline_path))

# Simulation timestep
idf.idfobjects['TIMESTEP'][0].Number_of_Timesteps_per_Hour = settings['timesteps_per_hour']

# Heating and cooling setpoints: constant values -> day/night Schedule:Compact
for name, fields in settings['setpoint_schedules'].items():
    for constant in [s for s in idf.idfobjects['SCHEDULE:CONSTANT'] if s.Name == name]:
        idf.removeidfobject(constant)
    schedule = idf.newidfobject('SCHEDULE:COMPACT', Name=name, Schedule_Type_Limits_Name='temperatureScheduleTypeLimit')
    for i, value in enumerate(fields, start=1):
        schedule[f'Field_{i}'] = value

# Always-on availability of heating, cooling and ventilation; lighting and equipment schedules
for schedule in idf.idfobjects['SCHEDULE:COMPACT']:
    if schedule.Name in settings['modified_schedules']:
        schedule.obj = schedule.obj[:3] + settings['modified_schedules'][schedule.Name]

# Equipment power density of the tuned zones
for equipment in idf.idfobjects['ELECTRICEQUIPMENT']:
    for density, zones in settings['electric_equipment_power_density_w_m2'].items():
        if equipment.Name.split('..')[1] in zones:
            equipment.Watts_per_Floor_Area = float(density)

# Outdoor air computed from the flow rate per floor area (CAV systems controlled by volume flow)
for outdoor_air in idf.idfobjects['DESIGNSPECIFICATION:OUTDOORAIR']:
    outdoor_air.Outdoor_Air_Method = settings['outdoor_air_method']

# External blinds
for blind in idf.idfobjects['WINDOWMATERIAL:BLIND']:
    for field, value in settings['external_blind_properties'].items():
        blind[field] = value

# Hourly output variables used for tuning, KPIs and analysis
for output_variable in list(idf.idfobjects['OUTPUT:VARIABLE']):
    idf.removeidfobject(output_variable)
idf = Eppy.IdfSetOutputVariables(idf, settings['output_variables'], 'Hourly')

idf.saveas(str(calibrated_path))
print(f'Tuned BEM saved to {calibrated_path}')

### 1.1 Check against the tuned BEM of the paper

Field-by-field comparison with `models/<building>_bem_calibrated.idf`.

In [ ]:
def idf_objects(idf_file):
    objects = {}
    for key, idf_objects_of_type in idf_file.idfobjects.items():
        for i, obj in enumerate(idf_objects_of_type):
            values = [str(v).strip().lower() for v in obj.obj[1:]]
            while values and values[-1] == '':
                values.pop()
            name = values[0] if len(obj.fieldnames) > 1 and 'name' in obj.fieldnames[1].lower() else i
            objects[(key.upper(), name)] = values
    return objects


def same_value(a, b):
    try:
        return abs(float(a) - float(b)) < 1e-6
    except ValueError:
        return a == b


tuned, archived = idf_objects(IDF(str(calibrated_path))), idf_objects(IDF(str(archived_calibrated_path)))
differences = sorted(str(k) for k in set(tuned) ^ set(archived))
differences += sorted(str(k) for k in set(tuned) & set(archived)
                      if len(tuned[k]) != len(archived[k]) or not all(same_value(a, b) for a, b in zip(tuned[k], archived[k])))
print(f'{len(differences)} objects differ from the archived tuned BEM')
print('\n'.join(differences[:20]))

## 2. Annual simulations of the baseline and tuned models

Both models are converted to EnergyPlus 24.2 and simulated with the weather file of the
on-site station (2024).

In [ ]:
MODELS = {'baseline': baseline_path, 'tuned': calibrated_path}

for model, path in MODELS.items():
    simulation_idf = simulation_dir / f'{BUILDING}_{model}.idf'
    if RUN_SIMULATIONS:
        # Run period: 1 January - 31 December 2024
        Eppy.IdfSetRunPeriod(IDF(str(path)), config.RUN_PERIOD).saveas(str(simulation_idf))
        Eppy.IdfTransitionVersion(idfFilePath=str(simulation_idf), transitionExePath=str(config.TRANSITION_V24_1_TO_V24_2))
        ok = Eppy.EnergyPlus_RunSimulation(idfFilePath=str(simulation_idf), epwFilePath=str(config.EPW_PATH),
                                           energyPlusExePath=str(config.ENERGYPLUS_EXE))
        print(f'{model}: {"simulation completed" if ok else "simulation FAILED"}')

## 3. Measured vs simulated indoor air temperature

In [ ]:
def energyplus_timestamps(date_time, year=2024):
    """EnergyPlus ' MM/DD  HH:MM:SS' (hour ending, 24:00 = midnight) -> datetime"""
    parts = date_time.str.strip().str.split(r'\s+', expand=True)
    hours = parts[1].str.slice(0, 2).astype(int)
    dates = pd.to_datetime(f'{year}/' + parts[0], format='%Y/%m/%d')
    return dates + pd.to_timedelta(hours, unit='h')


measured = pd.read_csv(config.MONITORING_DIR / 'indoor_air_temperature_monitored_rooms.csv', sep=';')
measured = pd.DataFrame({
    'timestamp': pd.to_datetime(measured['Date'] + ' ' + measured['Hour'], format='%d/%m/%Y %H:%M:%S'),
    'measured': measured[monitored_room['sensor_column']],
})

comparison = measured.copy()
for model in MODELS:
    results = pd.read_csv(simulation_dir / f'{BUILDING}_{model}.csv')
    column = f"{monitored_room['zone']}:{SIMULATED_TEMPERATURE} [C](Hourly)"
    simulated = pd.DataFrame({'timestamp': energyplus_timestamps(results['Date/Time']), model: results[column]})
    if model == 'tuned':
        # occupants of the classroom (booking schedule), used to shade the occupied hours in the figures
        simulated['occupants'] = results[f"{monitored_room['zone']}:Zone People Occupant Count [](Hourly)"]
    comparison = comparison.merge(simulated, on='timestamp', how='inner')
comparison = comparison.dropna()
comparison.to_csv(output_dir / f'{BUILDING}_hourly_temperature_comparison.csv', index=False)
print(f'{len(comparison)} hourly values compared')

### 3.1 Error metrics

For each period: mean temperature difference (MTDW, measured − simulated), relative error
(MTDW over the mean measured temperature), mean absolute error (MAE) and root mean square
error (RMSE). The representative weeks of Table 3 run from Monday 01:00 to the following
Monday 00:00.

In [ ]:
PERIODS = {
    'Full year': None,
    'Winter week': '2024-01-22',
    'Spring week': '2024-05-27',
    'Summer week': '2024-06-17',
    'Autumn week': '2024-11-25',
}


def period_data(df, week_start):
    if week_start is None:
        return df
    start = pd.Timestamp(week_start)
    return df[(df['timestamp'] > start) & (df['timestamp'] <= start + pd.Timedelta(days=7))]


def error_metrics(df, model):
    difference = df['measured'] - df[model]
    return {'n_hours': len(df),
            'MTDW_C': difference.mean(),
            'error_pct': difference.mean() / df['measured'].mean() * 100,
            'MAE_C': difference.abs().mean(),
            'RMSE_C': np.sqrt((difference ** 2).mean())}


rows = []
for period, week_start in PERIODS.items():
    for model in MODELS:
        rows.append({'building': BUILDING, 'room': monitored_room['room'], 'period': period, 'model': model,
                     **error_metrics(period_data(comparison, week_start), model)})
metrics = pd.DataFrame(rows)
metrics.to_csv(output_dir / f'{BUILDING}_calibration_metrics.csv', index=False)
metrics.pivot(index='period', columns='model', values=['MTDW_C', 'error_pct', 'MAE_C', 'RMSE_C']).round(2).loc[list(PERIODS)]

### 3.2 Representative weeks

In [ ]:
for period, week_start in list(PERIODS.items())[1:]:
    figure = px.line(period_data(comparison, week_start), x='timestamp', y=['measured', 'baseline', 'tuned'],
                     labels={'value': 'Indoor temperature (°C)', 'timestamp': ''},
                     title=f"{config.BUILDINGS[BUILDING]['name']}, room {monitored_room['room']}: {period.lower()}")
    config.show_figure(figure, f"{BUILDING}_{period.lower().replace(' ', '_')}", figures_dir)

### 3.3 Monthly means

Monthly mean measured and simulated temperatures and their difference (tuned model,
simulated − measured).

In [ ]:
monthly = comparison.groupby(comparison['timestamp'].dt.month)[['measured', 'baseline', 'tuned']].mean()
monthly['difference_tuned'] = monthly['tuned'] - monthly['measured']
monthly.index.name = 'month'
monthly.round(2).to_csv(output_dir / f'{BUILDING}_monthly_temperature_comparison.csv')

figure = px.bar(monthly.reset_index(), x='month', y='difference_tuned',
                labels={'difference_tuned': 'Simulated − measured (°C)'},
                title=f"{config.BUILDINGS[BUILDING]['name']}: monthly mean temperature difference after tuning")
config.show_figure(figure, f'{BUILDING}_monthly_temperature_difference', figures_dir)
monthly.round(2)